# Larger LLMs for schema discovery, smaller LLMs for application

We build one **Elastic Workflow** that asks: *what kind of pilot wrote this report?*
A large reasoning model reads a stratified sample of [NASA ASRS](https://asrs.arc.nasa.gov/) reports and proposes a categorical schema. A human approves it from the Kibana UI. A small [Mistral](https://mistral.ai/) model applies the schema across the full corpus.

## Setup

In [ ]:
%pip install elasticsearch==9.4 python-dotenv==1.0.1 pandas==2.2.3 requests==2.32.3 -q

In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

## Create the corpus index

Mappings match the ASRS columns we will use: `flight_phase` and `anomaly` for stratification, `narrative` and `synopsis` as document text.

In [3]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected to Elasticsearch: {es.info()['version']['number']}")

Connected to Elasticsearch: 9.4.1


In [4]:
INDEX = "incident_reports"

mappings = {
    "mappings": {
        "properties": {
            "acn": {"type": "keyword"},
            "flight_phase": {"type": "keyword"},
            "anomaly": {"type": "keyword"},
            "synopsis": {"type": "text"},
            "narrative": {"type": "text"},
        }
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)

es.indices.create(index=INDEX, body=mappings)
print(f"Created index: {INDEX}")

Created index: incident_reports


## Load ASRS reports

The dataset (`asrs.csv`) was extracted from the [NASA ASRS Database Online](https://asrs.arc.nasa.gov/search/database.html) using the public search interface and exported as CSV. It is bundled next to this notebook in the repository.

In [5]:
import pandas as pd

# ASRS CSV exports have a category row above the actual column names.
# Skip the category row; pandas drops the blank line that follows automatically.
df = pd.read_csv("asrs.csv", skiprows=[0], skip_blank_lines=True, low_memory=False)

# Some column names repeat (e.g. Aircraft 1/Aircraft 2 both have "Flight Phase");
# pandas renames duplicates with a ".1" suffix. We just use the first occurrence.
df = df.dropna(subset=["ACN", "Narrative"])


def to_doc(row):
    return {
        "_index": INDEX,
        "_id": str(row["ACN"]),
        "_source": {
            "acn": str(row["ACN"]),
            "flight_phase": row.get("Flight Phase"),
            "anomaly": row.get("Anomaly"),
            "synopsis": row.get("Synopsis"),
            "narrative": row.get("Narrative"),
        },
    }


ok, errors = helpers.bulk(
    es,
    (to_doc(r) for _, r in df.iterrows()),
    raise_on_error=False,
)
print(f"indexed {ok}, errors {len(errors)}")

indexed 4209, errors 78


## Register the Mistral inference endpoint

The executor step calls this endpoint once per document via `ai.agent`.

In [6]:
INFERENCE_ID = "mistral-small-extractor"

# Delete and re-create so the rate_limit setting picks up changes between runs.
# 6 RPM = 1 every 10s; conservative for Mistral free tier to avoid 429s.
es.options(ignore_status=[404]).inference.delete(inference_id=INFERENCE_ID)

es.inference.put(
    task_type="chat_completion",
    inference_id=INFERENCE_ID,
    inference_config={
        "service": "mistral",
        "service_settings": {
            "api_key": MISTRAL_API_KEY,
            "model": "mistral-small-latest",
            "rate_limit": {"requests_per_minute": 6},
        },
    },
)
print(f"Inference endpoint ready: {INFERENCE_ID}")

Inference endpoint ready: mistral-small-extractor


## Auxiliary indices

`schemas` stores one document per discovery run; `extractions` stores one document per source report per schema version.

In [7]:
for aux in ["schemas", "extractions"]:
    if not es.indices.exists(index=aux):
        es.indices.create(index=aux)
        print(f"Created index: {aux}")

## Create the workflow via the Workflows API

The workflow YAML is defined inline and registered via `POST /api/workflows`. Steps: stratified sample (by phase and by anomaly) > discover (large model) > human gate > store schema > fetch corpus > foreach over each report.

In [8]:
import requests

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

# Discover the GenAI connector ID (large reasoning model) from Kibana.
# Looks for any Bedrock / OpenAI / Gemini / generic GenAI connector.
GENAI_TYPES = {".bedrock", ".gen-ai", ".gemini", ".inference"}

response = requests.get(f"{KIBANA_URL}/api/actions/connectors", headers=headers)
response.raise_for_status()

genai_connectors = [c for c in response.json() if c["connector_type_id"] in GENAI_TYPES]
if not genai_connectors:
    raise RuntimeError(
        "No GenAI connectors found. Create one in Kibana > Stack Management > Connectors."
    )

KIBANA_CONNECTOR_ID = genai_connectors[0]["id"]
print(f"Using connector: {genai_connectors[0]['name']} ({KIBANA_CONNECTOR_ID})")

Using connector: Anthropic Claude Haiku 4.5 (Anthropic-Claude-Haiku-4-5)


In [9]:
WORKFLOW_NAME = "Pilot Schema Discovery and Application"

workflow_yaml = f"""name: {WORKFLOW_NAME}
description: >
  Discovers a categorical schema from a stratified sample of incident reports
  using a large reasoning model, then applies it across the corpus with a small
  Mistral model.
enabled: true

consts:
  corpusIndex: {INDEX}
  schemasIndex: schemas
  extractionsIndex: extractions

inputs:
  - name: goal
    type: string
    required: true
    default: "What kind of pilot wrote this report?"

triggers:
  - type: manual

steps:
  - name: by_phase
    type: elasticsearch.request
    with:
      method: POST
      path: "/{INDEX}/_search"
      body:
        size: 0
        aggs:
          per_phase:
            terms:
              field: flight_phase
              size: 8
            aggs:
              sampled:
                top_hits:
                  size: 5
                  _source: ["acn", "synopsis", "narrative"]

  - name: by_anomaly
    type: elasticsearch.request
    with:
      method: POST
      path: "/{INDEX}/_search"
      body:
        size: 0
        aggs:
          per_anomaly:
            terms:
              field: anomaly
              size: 8
            aggs:
              sampled:
                top_hits:
                  size: 3
                  _source: ["acn", "synopsis", "narrative"]

  - name: discover
    type: ai.prompt
    connector-id: "{KIBANA_CONNECTOR_ID}"
    with:
      systemPrompt: |
        You design categorical schemas for use by downstream classifiers. A
        schema is a small set of fields, each with a few mutually exclusive
        values. Every value you propose must be grounded in evidence from the
        provided sample and must serve the stated question. You do not invent
        values that are not supported by at least two documents in the sample.
        You do not propose fields that a reasonable analyst could have written
        without reading the documents.
      prompt: |
        Question:
        {{{{ inputs.goal }}}}

        Sample documents stratified by flight phase:
        {{{{ steps.by_phase.output.aggregations.per_phase.buckets | json }}}}

        Sample documents stratified by anomaly type:
        {{{{ steps.by_anomaly.output.aggregations.per_anomaly.buckets | json }}}}

        Propose between 2 and 4 categorical fields that:
        - serve the question (you can explain how)
        - depend on patterns visible in the sample (you can cite document IDs)
        - would not be obvious to someone who has not read the sample

        For each field, return: name (snake_case), definition, why_useful,
        and values (2 to 4 mutually exclusive options).

        For each value, return: value (snake_case) and definition.
      schema:
        type: object
        properties:
          fields:
            type: array
            minItems: 2
            maxItems: 4
            items:
              type: object
              required: [name, definition, why_useful, values]
              properties:
                name: {{ type: string }}
                definition: {{ type: string }}
                why_useful: {{ type: string }}
                values:
                  type: array
                  minItems: 2
                  maxItems: 4
                  items:
                    type: object
                    required: [value, definition]
                    properties:
                      value: {{ type: string }}
                      definition: {{ type: string }}
      temperature: 0.3

  - name: human_gate
    type: waitForInput
    with:
      message: "Review and edit the proposed schema. The approved fields will be applied across the full corpus."
      schema:
        type: object
        required: [approved_fields]
        properties:
          approved_fields:
            type: array
            items:
              type: object
              properties:
                name: {{ type: string }}
                definition: {{ type: string }}
                values:
                  type: array
                  items:
                    type: object
                    properties:
                      value: {{ type: string }}
                      definition: {{ type: string }}
          notes:
            type: string

  - name: store_schema
    type: elasticsearch.index
    with:
      index: "{{{{ consts.schemasIndex }}}}"
      document:
        question: "{{{{ inputs.goal }}}}"
        approved_fields: "{{{{ steps.human_gate.output.approved_fields }}}}"
        reviewer_notes: "{{{{ steps.human_gate.output.notes }}}}"

  - name: fetch_corpus
    type: elasticsearch.request
    with:
      method: POST
      path: "/{INDEX}/_search"
      body:
        size: 100
        _source: ["acn", "narrative"]
        query:
          match_all: {{}}

  - name: classify_all
    type: foreach
    foreach: "${{{{ steps.fetch_corpus.output.hits.hits }}}}"
    iteration-on-failure:
      retry:
        max-attempts: 5
        delay: "3s"
    steps:
      - name: classify
        type: ai.agent
        inference-id: "{INFERENCE_ID}"
        timeout: "120s"
        with:
          message: |
            You will classify the following report against a fixed schema.
            For each field in the schema, assign exactly one of its value
            options, or null if none of the values clearly applies. Include
            the short quote that supports the assignment and a confidence
            score between 0 and 1. Set review_required to true if any field
            returned null or any confidence is below 0.5.

            Schema:
            {{{{ steps.human_gate.output.approved_fields | json }}}}

            Report:
            {{{{ foreach.item._source.narrative }}}}
          schema:
            type: object
            properties:
              field_values:
                type: object
                additionalProperties: true
              review_required: {{ type: boolean }}
      - name: write_extraction
        type: elasticsearch.index
        with:
          index: "{{{{ consts.extractionsIndex }}}}"
          document:
            acn: "{{{{ foreach.item._source.acn }}}}"
            field_values: "{{{{ steps.classify.output.structured_output.field_values }}}}"
            review_required: "{{{{ steps.classify.output.structured_output.review_required }}}}"
"""

In [10]:
WORKFLOW_ID = "pilot-schema-discovery"
WORKFLOW_NAME = "Pilot Schema Discovery and Application"

# Delete any existing workflow with the same id (allows re-runs)
requests.delete(f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}", headers=headers)

# Bulk create. If the workflow already exists, the PUT below will update it.
response = requests.post(
    f"{KIBANA_URL}/api/workflows",
    headers=headers,
    json={"workflows": [{"id": WORKFLOW_ID, "yaml": workflow_yaml}]},
)
response.raise_for_status()

# PUT sets the display name, description, and enables the workflow.
put_response = requests.put(
    f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}",
    headers=headers,
    json={
        "name": WORKFLOW_NAME,
        "description": "Discovers a categorical schema from a stratified sample of incident reports, then applies it across the corpus with a small Mistral model.",
        "yaml": workflow_yaml,
        "enabled": True,
    },
)
put_response.raise_for_status()

print(f"Workflow ready: {WORKFLOW_ID} ({WORKFLOW_NAME})")
print(f"Open in Kibana: {KIBANA_URL}/app/workflows/{WORKFLOW_ID}")

Workflow ready: pilot-schema-discovery (Pilot Schema Discovery and Application)
Open in Kibana: https://4622216ea8cd443ead5bef0a3de05135.us-central1.gcp.cloud.es.io/app/workflows/pilot-schema-discovery


## Trigger the workflow

`POST /api/workflows/workflow/{id}/run` starts an execution and returns a `workflowExecutionId`. The run will pause at the `human_gate` step until the schema is approved from the Kibana UI (Workflows > Executions > the running one). After approval, the executor fans out and writes to the `extractions` index.

In [13]:
response = requests.post(
    f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}/run",
    headers=headers,
    json={"inputs": {"goal": "What kind of pilot wrote this report?"}},
)
response.raise_for_status()

execution_id = response.json()["workflowExecutionId"]
print(f"Execution started: {execution_id}")
print(
    f"Approve the schema at: {KIBANA_URL}/app/workflows/{WORKFLOW_ID}/executions/{execution_id}"
)

Execution started: a9701d13-dc80-406b-b93f-85aa839feb3c
Approve the schema at: https://4622216ea8cd443ead5bef0a3de05135.us-central1.gcp.cloud.es.io/app/workflows/pilot-schema-discovery/executions/a9701d13-dc80-406b-b93f-85aa839feb3c


## Print the discover step output

`waitForInput` cannot be pre-populated from a previous step today (the API exposes only `message` and `schema`). To make the human gate practical, we poll the execution until the `discover` step completes and pretty-print its output. Copy that JSON, edit it if needed, and paste it into the `approved_fields` field of the form.

In [12]:
import time


def step_output(execution_id, step_id, timeout=120, interval=3):
    """Poll an execution until the given step completes, then return its output."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        exec_response = requests.get(
            f"{KIBANA_URL}/api/workflows/executions/{execution_id}", headers=headers
        )
        exec_response.raise_for_status()
        step = next(
            (
                s
                for s in exec_response.json().get("stepExecutions", [])
                if s.get("stepId") == step_id
                and s.get("status") in ("completed", "failed")
            ),
            None,
        )
        if step:
            step_response = requests.get(
                f"{KIBANA_URL}/api/workflows/executions/{execution_id}/step/{step['id']}",
                headers=headers,
            )
            step_response.raise_for_status()
            return step_response.json().get("output")
        time.sleep(interval)
    raise TimeoutError(f"step '{step_id}' did not complete in time")


discover = step_output(execution_id, "discover")

# Strip `why_useful` (not part of the human_gate form) and wrap in the shape
# expected by the waitForInput form so this is paste-ready.
approved_fields = [
    {
        "name": field["name"],
        "definition": field["definition"],
        "values": [
            {
                "value": v["value"],
                "definition": v["definition"],
            }
            for v in field["values"]
        ],
    }
    for field in discover["content"]["fields"]
]

print(json.dumps({"approved_fields": approved_fields, "notes": ""}, indent=2))

{
  "approved_fields": [
    {
      "name": "pilot_experience_level",
      "definition": "The demonstrated level of operational experience and training status of the pilot who wrote the report, inferred from the type of aircraft operated, certification level, and decision-making patterns evident in the narrative.",
      "values": [
        {
          "value": "professional_air_carrier",
          "definition": "Captain or First Officer operating large transport category aircraft (B737, A320, CRJ, etc.) for scheduled air carrier operations, demonstrating structured crew resource management and adherence to formal procedures."
        },
        {
          "value": "experienced_general_aviation",
          "definition": "Pilot-in-command of general aviation aircraft (Cessna, Cirrus, experimental) with demonstrated experience in pattern work, cross-country operations, or flight instruction, showing consistent radio discipline and procedural knowledge."
        },
        {
          

## Cleanup

In [ ]:
for idx in [INDEX, "schemas", "extractions"]:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)

es.options(ignore_status=[404]).inference.delete(inference_id=INFERENCE_ID)

In [ ]:
requests.delete(f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}", headers=headers)
print(f"Deleted workflow: {WORKFLOW_ID}")